# Tratamento de Microdados do ENEM (2022 e 2023)

Este notebook estrutura o pipeline de tratamento dos microdados do ENEM referentes ao município de Curitiba. As etapas principais compreendem a categorização, filtragem, limpeza, seleção de features e a unificação das bases (2022 e 2023). O arquivo consolidado gerado ao final é exportado em formato CSV para a modelagem do dashboard no Looker Studio para compartilhamento.

In [23]:
import pandas as pd
import os

print(pd.__version__)

3.0.1


## 1. Categorização

In [24]:
colunas_desejadas = [
    'NU_INSCRICAO',
    'NU_ANO',
    'SG_UF_PROVA',
    'NO_MUNICIPIO_PROVA',
    'TP_SEXO',
    'TP_COR_RACA',
    'TP_ESCOLA',
    'IN_TREINEIRO',
    'TP_PRESENCA_CN',
    'TP_PRESENCA_CH',
    'TP_PRESENCA_LC',
    'TP_PRESENCA_MT',
    'NU_NOTA_CN',
    'NU_NOTA_CH',
    'NU_NOTA_LC',
    'NU_NOTA_MT',
    'NU_NOTA_REDACAO',
    'Q006'
]

mapa_escola = {
    1: 'Não informado (Egresso)',
    2: 'Pública',
    3: 'Privada'
}

mapa_cor_raca = {
    0: 'Não declarado',
    1: 'Branca',
    2: 'Preta',
    3: 'Parda',
    4: 'Amarela',
    5: 'Indígena',
    6: 'Não disposta'
}

mapa_sexo = {
    'M': 'Masculino',
    'F': 'Feminino'
}

mapa_renda = {
    'A': 'A - Nenhuma Renda',
    'B': 'B - Até 1 SM',
    'C': 'C - De 1 a 1,5 SM',
    'D': 'D - De 1,5 a 2 SM',
    'E': 'E - De 2 a 2,5 SM',
    'F': 'F - De 2,5 a 3 SM',
    'G': 'G - De 3 a 4 SM',
    'H': 'H - De 4 a 5 SM',
    'I': 'I - De 5 a 6 SM',
    'J': 'J - De 6 a 7 SM',
    'K': 'K - De 7 a 8 SM',
    'L': 'L - De 8 a 9 SM',
    'M': 'M - De 9 a 10 SM',
    'N': 'N - De 10 a 12 SM',
    'O': 'O - De 12 a 15 SM',
    'P': 'P - De 15 a 20 SM',
    'Q': 'Q - Mais de 20 SM'
}

def categorizar_grupo_renda(letra):
    if letra in ['A', 'B', 'C', 'D']:
        return 'Baixa Renda (Até R$ 2.424)'
    elif letra in ['E', 'F', 'G', 'H']:
        return 'Média Renda (R$ 2.424 a R$ 6.060)'
    elif letra in ['I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q']:
        return 'Alta Renda (Acima de R$ 6.060)'
    return 'Não informado'



In [25]:
print(f"colunas selecionadas: {len(colunas_desejadas)}")
print(f"escola código 1: {mapa_escola[1]}")
print(f"renda código B: {mapa_renda['B']}")
print(f"grupo renda código B {categorizar_grupo_renda('A')}")

colunas selecionadas: 18
escola código 1: Não informado (Egresso)
renda código B: B - Até 1 SM
grupo renda código B Baixa Renda (Até R$ 2.424)


## 2. Filtro "Curitiba"

### 2.1 Dados de 2022

In [26]:
caminho_2022 = 'microdados_enem_2022/DADOS/MICRODADOS_ENEM_2022.csv'
df_2022 = pd.read_csv(caminho_2022, sep=';', encoding='latin1', usecols=colunas_desejadas)

df_curitiba_2022 = df_2022[
    (df_2022['SG_UF_PROVA'] == 'PR') & 
    (df_2022['NO_MUNICIPIO_PROVA'] == 'Curitiba')
].copy()


del df_2022

print(f"inscritos em Curitiba (2022): {len(df_curitiba_2022):,}")

df_curitiba_2022.head(5)

inscritos em Curitiba (2022): 34,771


,NU_INSCRICAO,NU_ANO,TP_SEXO,TP_COR_RACA,TP_ESCOLA,IN_TREINEIRO,NO_MUNICIPIO_PROVA,SG_UF_PROVA,TP_PRESENCA_CN,TP_PRESENCA_CH,TP_PRESENCA_LC,TP_PRESENCA_MT,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,Q006
297,210055278087,2022,F,1,1,0,Curitiba,PR,1,1,1,1,544.1,519.9,486.1,590.4,640.0,H
387,210055565552,2022,F,1,1,0,Curitiba,PR,1,1,1,1,509.7,555.6,562.2,550.2,580.0,C
520,210055695328,2022,F,1,2,0,Curitiba,PR,1,1,1,1,560.2,579.6,617.2,557.0,560.0,E
552,210054733453,2022,F,1,1,0,Curitiba,PR,0,0,0,0,NaN,NaN,NaN,NaN,NaN,H
588,210055029252,2022,F,1,2,0,Curitiba,PR,1,1,1,1,516.3,527.9,586.5,683.7,720.0,K


### 2.2 Dados de 2023

In [27]:
caminho_2023 = 'microdados_enem_2023/DADOS/MICRODADOS_ENEM_2023.csv'

df_2023 = pd.read_csv(caminho_2023, sep=';', encoding='latin1', usecols=colunas_desejadas)

df_curitiba_2023 = df_2023[
    (df_2023['SG_UF_PROVA'] == 'PR') & 
    (df_2023['NO_MUNICIPIO_PROVA'] == 'Curitiba')
].copy()

del df_2023

print(f"Total de inscritos em Curitiba (2023): {len(df_curitiba_2023):,}")

df_curitiba_2023.head(5)

Total de inscritos em Curitiba (2023): 37,095


,NU_INSCRICAO,NU_ANO,TP_SEXO,TP_COR_RACA,TP_ESCOLA,IN_TREINEIRO,NO_MUNICIPIO_PROVA,SG_UF_PROVA,TP_PRESENCA_CN,TP_PRESENCA_CH,TP_PRESENCA_LC,TP_PRESENCA_MT,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,Q006
339,210059980959,2023,F,3,1,0,Curitiba,PR,0,0,0,0,NaN,NaN,NaN,NaN,NaN,B
461,210060214102,2023,F,1,2,0,Curitiba,PR,1,1,1,1,557.0,518.8,593.9,492.6,660.0,B
795,210058387363,2023,F,1,2,0,Curitiba,PR,1,1,1,1,501.5,520.8,582.9,571.0,860.0,G
835,210060441017,2023,F,1,2,0,Curitiba,PR,1,1,1,1,533.7,570.2,564.2,506.0,620.0,D
1243,210059899854,2023,F,1,1,0,Curitiba,PR,0,0,0,0,NaN,NaN,NaN,NaN,NaN,D


## 3. Limpeza

In [28]:
display(df_curitiba_2022.isnull().sum())

NU_INSCRICAO              0
NU_ANO                    0
TP_SEXO                   0
TP_COR_RACA               0
TP_ESCOLA                 0
IN_TREINEIRO              0
NO_MUNICIPIO_PROVA        0
SG_UF_PROVA               0
TP_PRESENCA_CN            0
TP_PRESENCA_CH            0
TP_PRESENCA_LC            0
TP_PRESENCA_MT            0
NU_NOTA_CN            10759
NU_NOTA_CH             9292
NU_NOTA_LC             9292
NU_NOTA_MT            10759
NU_NOTA_REDACAO        9292
Q006                      0
dtype: int64

In [29]:
display(df_curitiba_2023.isnull().sum())

NU_INSCRICAO              0
NU_ANO                    0
TP_SEXO                   0
TP_COR_RACA               0
TP_ESCOLA                 0
IN_TREINEIRO              0
NO_MUNICIPIO_PROVA        0
SG_UF_PROVA               0
TP_PRESENCA_CN            0
TP_PRESENCA_CH            0
TP_PRESENCA_LC            0
TP_PRESENCA_MT            0
NU_NOTA_CN            10769
NU_NOTA_CH             9380
NU_NOTA_LC             9380
NU_NOTA_MT            10769
NU_NOTA_REDACAO        9380
Q006                      0
dtype: int64

In [30]:
dup_2022 = df_curitiba_2022.duplicated(subset=['NU_INSCRICAO']).sum()
print({dup_2022})

{np.int64(0)}


In [31]:
dup_2023 = df_curitiba_2023.duplicated(subset=['NU_INSCRICAO']).sum()
print({dup_2023})

{np.int64(0)}


In [32]:
# Remover nulos
colunas_notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

df_curitiba_2022 = df_curitiba_2022[
    (df_curitiba_2022['TP_PRESENCA_CN'] == 1) &
    (df_curitiba_2022['TP_PRESENCA_CH'] == 1) &
    (df_curitiba_2022['TP_PRESENCA_LC'] == 1) &
    (df_curitiba_2022['TP_PRESENCA_MT'] == 1)
].dropna(subset=colunas_notas)

df_curitiba_2023 = df_curitiba_2023[
    (df_curitiba_2023['TP_PRESENCA_CN'] == 1) &
    (df_curitiba_2023['TP_PRESENCA_CH'] == 1) &
    (df_curitiba_2023['TP_PRESENCA_LC'] == 1) &
    (df_curitiba_2023['TP_PRESENCA_MT'] == 1)
].dropna(subset=colunas_notas)

# Remover treineiros
df_curitiba_2022 = df_curitiba_2022[df_curitiba_2022['IN_TREINEIRO'] == 0]
df_curitiba_2023 = df_curitiba_2023[df_curitiba_2023['IN_TREINEIRO'] == 0]

In [33]:
print(f"candidatos reais em 2022: {len(df_curitiba_2022):,}")
print(f"candidatos reais em 2023: {len(df_curitiba_2023):,}")

candidatos reais em 2022: 19,942
candidatos reais em 2023: 21,542


## 4. Feature engineering

In [34]:
def features(df):
    # Feature numérica: média
    df['NU_NOTA_GERAL'] = (
        df['NU_NOTA_CN'] + df['NU_NOTA_CH'] + 
        df['NU_NOTA_LC'] + df['NU_NOTA_MT'] + df['NU_NOTA_REDACAO']
    ) / 5.0
    df['NU_NOTA_GERAL'] = df['NU_NOTA_GERAL'].round(2)
    
    # Features categóricas: mapeamento
    df['TP_ESCOLA_DESC'] = df['TP_ESCOLA'].map(mapa_escola)
    df['TP_COR_RACA_DESC'] = df['TP_COR_RACA'].map(mapa_cor_raca)
    df['TP_SEXO_DESC'] = df['TP_SEXO'].map(mapa_sexo)
    df['Q006_RENDA_DESC'] = df['Q006'].map(mapa_renda)
    df['GRUPO_RENDA'] = df['Q006'].apply(categorizar_grupo_renda)
    
    return df

# Aplicar a função nos dois dfs
df_curitiba_2022 = features(df_curitiba_2022)
df_curitiba_2023 = features(df_curitiba_2023)

display(df_curitiba_2022[['NU_INSCRICAO', 'TP_ESCOLA_DESC', 'TP_COR_RACA_DESC', 'GRUPO_RENDA', 'NU_NOTA_GERAL']].head(5))


,NU_INSCRICAO,TP_ESCOLA_DESC,TP_COR_RACA_DESC,GRUPO_RENDA,NU_NOTA_GERAL
297,210055278087,Não informado (Egresso),Branca,Média Renda (R$ 2.424 a R$ 6.060),556.10
387,210055565552,Não informado (Egresso),Branca,Baixa Renda (Até R$ 2.424),551.54
520,210055695328,Pública,Branca,Média Renda (R$ 2.424 a R$ 6.060),574.80
588,210055029252,Pública,Branca,Alta Renda (Acima de R$ 6.060),606.88
863,210055497557,Pública,Parda,Baixa Renda (Até R$ 2.424),519.82


## 5. Concatenação

In [35]:
df_concat = pd.concat([df_curitiba_2022, df_curitiba_2023], ignore_index=True)

colunas_finais = [
    'NU_INSCRICAO', 'NU_ANO', 'NO_MUNICIPIO_PROVA',
    'TP_SEXO_DESC', 'TP_COR_RACA_DESC', 'TP_ESCOLA_DESC',
    'Q006', 'Q006_RENDA_DESC', 'GRUPO_RENDA',
    'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO',
    'NU_NOTA_GERAL'
]

df_final = df_concat[colunas_finais]
print(f"registros agrupados: {len(df_final):,}")
df_final.head(5)

registros agrupados: 41,484


,NU_INSCRICAO,NU_ANO,NO_MUNICIPIO_PROVA,TP_SEXO_DESC,TP_COR_RACA_DESC,TP_ESCOLA_DESC,Q006,Q006_RENDA_DESC,GRUPO_RENDA,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,NU_NOTA_GERAL
0,210055278087,2022,Curitiba,Feminino,Branca,Não informado (Egresso),H,H - De 4 a 5 SM,Média Renda (R$ 2.424 a R$ 6.060),544.1,519.9,486.1,590.4,640.0,556.10
1,210055565552,2022,Curitiba,Feminino,Branca,Não informado (Egresso),C,"C - De 1 a 1,5 SM",Baixa Renda (Até R$ 2.424),509.7,555.6,562.2,550.2,580.0,551.54
2,210055695328,2022,Curitiba,Feminino,Branca,Pública,E,"E - De 2 a 2,5 SM",Média Renda (R$ 2.424 a R$ 6.060),560.2,579.6,617.2,557.0,560.0,574.80
3,210055029252,2022,Curitiba,Feminino,Branca,Pública,K,K - De 7 a 8 SM,Alta Renda (Acima de R$ 6.060),516.3,527.9,586.5,683.7,720.0,606.88
4,210055497557,2022,Curitiba,Feminino,Parda,Pública,C,"C - De 1 a 1,5 SM",Baixa Renda (Até R$ 2.424),481.9,536.7,498.6,521.9,560.0,519.82


## 6. Exportação

In [36]:
#os.makedirs('dados', exist_ok=True)

output_path = os.path.join('dados', 'dados_enem_curitiba_2022_2023.csv')
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

tamanho_mb = os.path.getsize(output_path) / (1024 * 1024)

print(f"registros: {len(df_final):,}")
print(f"tamanho do arquivo: {tamanho_mb:.2f} MB")

df_final.head(5)


registros: 41,484
tamanho do arquivo: 5.87 MB


,NU_INSCRICAO,NU_ANO,NO_MUNICIPIO_PROVA,TP_SEXO_DESC,TP_COR_RACA_DESC,TP_ESCOLA_DESC,Q006,Q006_RENDA_DESC,GRUPO_RENDA,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,NU_NOTA_GERAL
0,210055278087,2022,Curitiba,Feminino,Branca,Não informado (Egresso),H,H - De 4 a 5 SM,Média Renda (R$ 2.424 a R$ 6.060),544.1,519.9,486.1,590.4,640.0,556.10
1,210055565552,2022,Curitiba,Feminino,Branca,Não informado (Egresso),C,"C - De 1 a 1,5 SM",Baixa Renda (Até R$ 2.424),509.7,555.6,562.2,550.2,580.0,551.54
2,210055695328,2022,Curitiba,Feminino,Branca,Pública,E,"E - De 2 a 2,5 SM",Média Renda (R$ 2.424 a R$ 6.060),560.2,579.6,617.2,557.0,560.0,574.80
3,210055029252,2022,Curitiba,Feminino,Branca,Pública,K,K - De 7 a 8 SM,Alta Renda (Acima de R$ 6.060),516.3,527.9,586.5,683.7,720.0,606.88
4,210055497557,2022,Curitiba,Feminino,Parda,Pública,C,"C - De 1 a 1,5 SM",Baixa Renda (Até R$ 2.424),481.9,536.7,498.6,521.9,560.0,519.82


## 7. Análise inicial

In [37]:
print("Média da Nota Geral em Curitiba por categoria:")
df_final.groupby(['NU_ANO', 'TP_ESCOLA_DESC'])['NU_NOTA_GERAL'].mean().round(2).unstack()


Média da Nota Geral em Curitiba por categoria:


TP_ESCOLA_DESC,Não informado (Egresso),Privada,Pública
NU_ANO,,,
2022,586.04,613.52,541.48
2023,584.18,623.69,537.37


Estudante da rede pública parece ter uma média menor (-86 pontos em 2023), o que é significativo se pensar no SISU, podendo ser uma barreira de acesso à universidade. E em 2023, a nota geral da categoria Privada aumentou enquanto a da Pública diminuiu.